# Rush Impact (Polars)

Compares weekday rush-window performance against non-rush periods using the Polars analysis path.

In [1]:
from pathlib import Path
import importlib.util
import sys

import plotly.express as px
import polars as pl

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "pyproject.toml").exists() and (candidate / "analysis" / "polars").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("Run this notebook from the project root or a subdirectory.")

POLARS_ANALYSIS_DIR = PROJECT_ROOT / "analysis" / "polars"
polars_analysis_path = str(POLARS_ANALYSIS_DIR)
if polars_analysis_path in sys.path:
    sys.path.remove(polars_analysis_path)
sys.path.insert(0, polars_analysis_path)

for module_name in ("_shared", "report_cache", "cli_common"):
    module = sys.modules.get(module_name)
    module_file = getattr(module, "__file__", None) if module else None
    module_path = Path(module_file).resolve() if module_file else None
    if module_path is not None and POLARS_ANALYSIS_DIR not in module_path.parents:
        sys.modules.pop(module_name, None)

def load_polars_script(module_name: str, filename: str):
    spec = importlib.util.spec_from_file_location(module_name, POLARS_ANALYSIS_DIR / filename)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module

rush_impact = load_polars_script("polars_rush_impact", "rush-impact.py")

DB = PROJECT_ROOT / "data" / "foli.db"
CACHE_DIR = PROJECT_ROOT / "outputs" / "polars-report-cache"
FORCE_CACHE = False
NO_CACHE = False
TIMEZONE = "Europe/Helsinki"
RUSH_WINDOWS = ["07:00-09:00", "15:00-18:00"]
INCLUDE_WEEKENDS = False
MIN_OBSERVATIONS = 50
LIMIT = 10
QUALITY_MODE = "conservative"
BUCKET = "trip-stop"


Set `NO_CACHE = True` to read SQLite directly, or `FORCE_CACHE = True` to rebuild the Polars cache.

In [2]:
class Args:
    db = DB
    cache_dir = CACHE_DIR
    force_cache = FORCE_CACHE
    no_cache = NO_CACHE
    timezone = TIMEZONE
    rush_window = RUSH_WINDOWS
    include_weekends = INCLUDE_WEEKENDS
    min_observations = MIN_OBSERVATIONS
    limit = LIMIT
    quality_mode = QUALITY_MODE
    exclude_stop_call_disagreement = False
    bucket = BUCKET

buckets = rush_impact.load_delay_buckets_for_args(Args)
impact = rush_impact.build_rush_impact(
    buckets,
    rush_windows=tuple(rush_impact.rush_window_values(RUSH_WINDOWS)),
    include_weekends=INCLUDE_WEEKENDS,
    min_observations=MIN_OBSERVATIONS,
    limit=LIMIT,
)
impact


line_ref,line_name,bucket_count_non_rush,bucket_count_rush,raw_poll_count_non_rush,raw_poll_count_rush,median_delay_min_non_rush,median_delay_min_rush,rush_median_delay_lift_min,p90_delay_min_non_rush,p90_delay_min_rush,rush_p90_delay_lift_min,pct_over_5_min_late_non_rush,pct_over_5_min_late_rush,rush_over_5_min_late_pct_point_lift
str,str,u32,u32,i64,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""612""","""612""",422,975,915,2134,0.22,7.48,7.26,4.27,16.55,12.27,6.16,64.62,58.45
"""615""","""615""",1014,2185,1891,4277,1.25,4.57,3.31,6.7,16.14,9.44,16.37,47.96,31.59
"""72""","""72""",2146,1138,3683,2034,1.34,4.1,2.76,5.67,10.87,5.21,13.79,41.21,27.42
"""721""","""721""",1428,774,3528,1532,1.12,3.52,2.41,4.3,8.03,3.73,7.0,37.6,30.59
"""701""","""701""",1011,2149,2849,6090,0.13,1.5,1.37,2.8,6.39,3.59,3.96,15.96,12.0
"""220""","""220""",33634,8841,84631,23140,0.41,2.89,2.48,3.89,7.37,3.47,5.09,25.98,20.89
"""77""","""77""",886,1306,2156,2324,-4.22,0.24,4.47,-0.56,2.83,3.39,1.13,0.84,-0.29
"""903""","""903""",315,680,1590,1989,-0.7,0.0,0.7,3.74,7.05,3.31,5.08,18.82,13.74
"""P1""","""P1""",574,290,2965,1593,-0.53,0.0,0.53,1.4,4.25,2.85,1.22,8.97,7.75


In [3]:
if impact.is_empty():
    print("No matching observations found.")
else:
    fig = px.bar(
        impact.sort("rush_p90_delay_lift_min"),
        x="rush_p90_delay_lift_min",
        y="line_ref",
        orientation="h",
        title="Rush delay lift by line",
        labels={"line_ref": "Line", "rush_p90_delay_lift_min": "Rush minus non-rush P90 delay, minutes"},
    )
    fig.update_layout(showlegend=False)
    fig.show()
